<a href="https://colab.research.google.com/github/HoGentTIN/ml-workshop-middelbaar-2627/blob/main/notebooks/LineaireRegressie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lineaire Regressie

Deze notebook toont de essentie van **gradient descent** a.d.h.v. **enkelvoudige lineaire regressie**.

In [ ]:
# Importeer de bibliotheken die we gebruiken.
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def genereer_data(m=50, low=-5, high=5, sigma=0.5, seed=42):
    """ Genereer m datapunten die ongeveer een lineair verband hebben
        met een willekeurige richtingscoëfficiënt en bias.

        m    : aantal datapunten
        low  : laagste x-waarde
        high : hoogste x-waarde
        sigma: standaardafwijking van de Gaussiaanse ruis
        seed : bepaalt de gegenereerde random waarden

        returns: x, y, a, b met x en y vectoren met n elementen terwijl a en b 
                 de "geheime" richtingscoëfficiënt en bias zijn
    """
    rng = np.random.default_rng(seed=seed)
    a = rng.uniform(-2, 2, size=1)[0]
    b = rng.uniform(2, 5, size=1)[0]
    
    x = np.linspace(low,high,50)
    y = a*x + b + rng.normal(loc=0, scale=sigma, size=m)
    
    return x, y, a, b

def plot_data(x, y, a=None, b=None):   
    """ Plot data, al dan niet met een rechte erbij.
    """
    fig, ax = plt.subplots()
    ax.scatter(x, y, alpha=0.5)
    ax.set_xticks([])
    ax.set_yticks([])    
    if a is not None:
        xs = np.linspace(min(x), max(x), 100)
        ys = a * xs + b
        ax.plot(xs, ys, color='red')

We genereren een aantal datapunten $(x_i, y_i)$ die ongeveer een lineair verband hebben, 
t.t.z.
$$
y \approx a \times x + b
$$
voor een onbekende $a$ en $b$ en we plotten deze data. Om het moeilijker te maken om de $a$ en $b$ met het blote oog in te schatten verwijderen we de schaal op de $X$- en $Y$-as.

In [ ]:
x, y, a_geheim, b_geheim = genereer_data(sigma=1,seed=42)
plot_data(x, y);

Plot de horizontale rechte 
$$
y = 0
$$
over de datapunten.

In [ ]:
plot_data(x, y, a=0, b=0)

## Opdracht 1: Experimenteer met $a$ en $b$

Experimenteer met verschillende waarden voor $a$ en $b$ om een "goed passende" rechte te vinden.

Probeer minstens 3 verschillende combinaties van $a$ en $b$ en noteer wat volgens jou 
de beste en slechtste combinatie is.

In [ ]:
plot_data(x, y)

Mijn drie combinaties zijn:

- 
-
-

## Opdracht 2: Bereken de kost voor deze verschillende $a$ en $b$ waarden

Met het blote oog kunnen we misschien wel ongeveer
inschatten hoe goed een rechte past bij de gegeven punten 
maar het is niet altijd zo gemakkelijk om verschillende 
rechten met elkaar te vergelijken.

Daarom definiëren we een wiskundige formule die in een getal uitdrukt hoe "goed" de rechte bij de gegeven datapunten past.
Hoe kleiner dit getal het beter de rechte bij de gegeven datapunten past.

We stellen
$$
\hat y_i = a \times x_i + b
$$
Dus bv. als $a = 2$ en $b=-1$, en $x_i = 3$, dan is 
$$
\hat y_i = 2 \times 3 - 1 = 5.
$$

Veronderstel nu dat het juiste antwoord $y_i$ gelijk is aan 7, dan zeggen we dat de **kost** voor
dit voorbeeld gelijk is aan 
$$
(\hat y_i - y_i)^2 = (5 - 7)^2 = (-2)^2 = 4. 
$$



In [ ]:
def figuur_kost_illustratie():
    """ Creëer een figuur om de kost voor een enkel datapunt te illustreren.
    """
    fig, ax = plt.subplots()
    x = np.linspace(0,2,10)
    rng = np.random.default_rng(seed=42)
    y = 2*x + 1 + rng.normal(loc=0, scale=1.5, size=10)
    ax.scatter(x, y, alpha=0.5)      
    xs = np.linspace(min(x), max(x), 100)
    ys = 2 * xs + 1
    ax.plot(xs, ys, color='red')
    
    xi = x[5]
    yi = 2*xi+1
    ax.plot(xi, yi, color='black', marker="+")
    ax.vlines([xi], ymin=y[5], ymax=yi, color='black')
    ax.text(xi+0.05, (yi+y[5])/2, "(vkw) kost", size="x-large")
    ax.text(xi+0.05, y[5], "$y_i$", size="x-large" )
    ax.text(xi, yi+0.25, r"$\hat y_i$", size="x-large")
        
figuur_kost_illustratie();

De volledige kost is dan de som (of eigenlijk het gemiddelde) van de kosten voor alle punten.
$$
\text{kost}(a,b) = \frac{1}{m}\sum_{i=1}^{m} (\hat y_i - y_i)^2 = \frac{1}{m}\sum_{i=1}^{m} (a\times x_i + b - y_i)^2
$$

De kost hangt dus af van $a$ en $b$: als we $a$ en $b$ variëren, dan krijgen we andere waarden voor de kost. 
De officiële naam voor deze kost is de **Mean Squared Error**, dikwijls afgekort tot MSE.

In [ ]:
def kost(x, y, a, b):
    """ Bereken de kost voor de gegeven datapunten x en y, en de coëfficiënten 
        a en b. De assumptie is straks dat x en y vast zijn en dat a en b 
        variëren.
    """
    yhat = a*x + b             # Bereken de voorspelling
    residuals = (yhat-y)**2    # Bereken de kwadraten van de verschillen
    return np.mean(residuals)  # Bereken het gemiddelde

Bij wijze van voorbeeld berekenen we de kost voor $a=0$ en $b=0$. 

Merk op: op zijn eentje betekent dit getal niet zo veel, je moet het vooral gaan vergelijken met de kost voor andere combinaties van $a$ en $b$.

In [ ]:
kost(x, y, a=0,b=0)

Bereken de kost voor de drie combinaties van $a$ en $b$ die je hierboven hebt opgeschreven. 
Komt de kost overeen met je intuïtie van wat de beste en slechtste combinatie was?

De drie berekende kosten waren:

- 
- 
- 

# Opdracht 3: De computer laten zoeken naar de beste $a$ en $b$

In opdracht 1 zocht je zelf een goede rechte door te proberen. Dat is traag, en je weet nooit zeker of het nog beter kan.
De computer kan dit automatisch met een techniek die **gradient descent** heet.


Stel je voor dat je op een berg staat in dichte mist en je wil zo laag mogelijk geraken.
Je ziet niets, maar met je voeten voel je wel welke kant het **bergaf** gaat.
Je zet een stapje in die richting, voelt opnieuw, zet weer een stapje, ... tot het nergens nog verder naar beneden gaat.

| Op de berg | Bij lineaire regressie |
|---|---|
| je positie | de waarden van $a$ en $b$ |
| de hoogte | de **kost**: hoe slecht de rechte past |
| het dal | de beste $a$ en $b$ |
| de grootte van je stappen | de **stapgrootte** |

## Stap voor stap

We gebruiken de bibliotheek **scikit-learn**. Het model `SGDRegressor` zoekt de beste rechte met gradient descent.
Met `eta0` kies je de stapgrootte.

In [ ]:
from sklearn.linear_model import SGDRegressor

X = x.reshape(-1, 1)   # scikit-learn wil de x-waarden als één kolom
model = SGDRegressor(learning_rate="constant", eta0=0.003)
ronde = 0

Elke keer dat je de cel hieronder uitvoert, doet het model **één ronde**: het bekijkt alle punten één keer
en past $a$ en $b$ telkens een klein beetje aan, in de richting waarin de kost daalt.

**Opdracht:** voer de cel hieronder **meerdere keren** na elkaar uit (met **Ctrl + Enter** blijf je in dezelfde cel).
Kijk hoe de rechte verschuift en hoe de kost daalt.

In [ ]:
model.partial_fit(X, y)   # één ronde gradient descent
ronde += 1

a, b = model.coef_[0], model.intercept_[0]
print(f"Ronde {ronde}: a = {a:.3f}, b = {b:.3f}, kost = {kost(x, y, a, b):.3f}")
plot_data(x, y, a, b)

- Na hoeveel rondes verandert de rechte bijna niet meer?
- Wat is de kost dan? Is die lager dan de kost van jouw beste combinatie uit opdracht 2?

## De stapgrootte

Hieronder trainen we het model drie keer, telkens 20 rondes, maar met een andere stapgrootte.
De grafiek toont de kost na elke ronde.

In [ ]:
for stapgrootte in [0.0001, 0.003, 0.1]:
    proefmodel = SGDRegressor(learning_rate="constant", eta0=stapgrootte)
    kosten = []
    for _ in range(20):
        proefmodel.partial_fit(X, y)
        kosten.append(kost(x, y, proefmodel.coef_[0], proefmodel.intercept_[0]))
    plt.plot(range(1, 21), kosten, "o-", label=f"stapgrootte {stapgrootte}")

plt.xlabel("ronde")
plt.ylabel("kost")
plt.legend()
plt.show()

**Opdracht:** wat merk je bij elke stapgrootte? 

**De keuze van de stapgrootte is dus van cruciaal belang bij gradient descent!**

Tot slot vergelijken we de gevonden $a$ en $b$ met de geheime waarden.

In [ ]:
print(f"De geschatte waarde van a: {a:1.5f}, de geheime waarde van a: {a_geheim:1.5f}")
print(f"De geschatte waarde van b: {b:1.5f}, de geheime waarde van b: {b_geheim:1.5f}")

## Wat heeft dit met neurale netwerken te maken?

Een neuraal netwerk wordt op **precies dezelfde manier** getraind:

1. een **model** met parameters (hier $a$ en $b$);
2. een **kost** die zegt hoe slecht het model het doet;
3. **gradient descent** dat ronde na ronde betere parameters zoekt.

Het enige verschil: het CNN voor steen, papier, schaar heeft geen 2 maar ongeveer **430 000** parameters.